# ECE, Brier and confidence intervals for the thesis tables
Reads the `results.json` files the evaluation already wrote (no model is run, **no test set is opened again**) and builds tables of mean over seeds with 95% intervals, for the group-split models and, if present, the original run.

**Before running:** add as inputs the output of **`evaluate_on_group_split`** (it has `v2/runs/arm_*/seed*/results.json`) and, optionally, the output of your original run (`runs/arm_*/seed*/results.json`). CPU is enough; it takes seconds.

**What each number is**
* AUROC, PR-AUC, sensitivity: early DR (grades 1-2) vs No DR, with a **patient-level bootstrap 95% interval** computed for each seed. The table shows the mean over seeds of the estimate and of the interval ends, and the sd of the estimate across seeds.
* ECE and Brier: calibration of **any DR vs No DR on all images** (the evaluation computes them on all grades), after temperature scaling and prior-shift correction. Lower is better.
* The interval is per-seed averaged, not pooled across seeds, so it describes image sampling noise; the seed sd describes training noise.

**What to send me:** the printed tables.

In [ ]:
import glob, json, os
import numpy as np, pandas as pd
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
files = glob.glob(os.path.join(INPUT, "**", "runs", "arm_*", "seed*", "results.json"), recursive=True)
if not files:
    raise SystemExit("STOP: add the output of evaluate_on_group_split (and optionally your original run) as an input.")
rows = []
for f in files:
    seed_dir = os.path.dirname(f); arm = os.path.basename(os.path.dirname(seed_dir)); seed = int(os.path.basename(seed_dir)[4:])
    run = "group split" if os.sep + "v2" + os.sep in f else "original"
    res = json.load(open(f))
    for source, r in res.items():
        e = r.get("early_vs_none")
        if not e: continue
        cal = r.get("calibration", {})
        rows.append({"run": run, "arm": arm, "seed": seed, "source": source,
                     "auroc": e["auroc"], "auroc_lo": e["auroc_ci"][0], "auroc_hi": e["auroc_ci"][1],
                     "pr_auc": e["pr_auc"], "pr_auc_lo": e["pr_auc_ci"][0], "pr_auc_hi": e["pr_auc_ci"][1],
                     "sensitivity": e["sensitivity"], "sensitivity_lo": e["sensitivity_ci"][0], "sensitivity_hi": e["sensitivity_ci"][1],
                     "specificity": e["specificity"], "ece": cal.get("ece"), "brier": cal.get("brier")})
raw = pd.DataFrame(rows)
print(raw.groupby(["run", "arm"]).seed.unique())
os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)
raw.to_csv(os.path.join(WORK, "audit", "metrics_per_seed.csv"), index=False)

## Mean over seeds with 95% intervals

In [ ]:
def fmt(g, m):
    mean = g[m].mean()
    if m + "_lo" in g:
        return f"{mean:.3f} ({g[m + '_lo'].mean():.3f}-{g[m + '_hi'].mean():.3f})"
    return f"{mean:.3f}"
out = []
for (run, source, arm), g in raw.groupby(["run", "source", "arm"]):
    out.append({"run": run, "source": source, "arm": arm, "seeds": len(g),
                "AUROC (95% CI)": fmt(g, "auroc"), "PR-AUC (95% CI)": fmt(g, "pr_auc"),
                "sensitivity (95% CI)": fmt(g, "sensitivity"), "specificity": fmt(g, "specificity"),
                "ECE": fmt(g, "ece"), "Brier": fmt(g, "brier"), "AUROC sd over seeds": round(g.auroc.std(), 3)})
table = pd.DataFrame(out)
table.to_csv(os.path.join(WORK, "audit", "metrics_with_ci.csv"), index=False)
for (run, source), g in table.groupby(["run", "source"]):
    print(f"\n=== {run} | {source}")
    print(g.drop(columns=["run", "source"]).to_string(index=False))

## How to read it
* Non-overlapping AUROC intervals between arms on a source mean the difference is larger than image-sampling noise; overlapping intervals do not prove they are equal.
* ECE near 0 and a low Brier score mean the probabilities can be taken at face value on that source. A large ECE on a source other than DDR shows the calibration, like the threshold, does not transfer.
* The calibration numbers cover any DR vs No DR on all grades, not only early DR.